# Steuerung — KI-gestützte Dokumentenaufbereitung

**Referenz-Implementierung · Niveau DQR 5/6**

Dieses Notebook ist der **einzige Einstiegspunkt** der Pipeline. Die Module
`pfade.py`, `schema.py`, `layout.py`, `erkennung.py`, `stapel.py`,
`kanonisch.py`, `sichtung.py` und `lauf.py` liegen unter
`python/pdf_extraction/` und werden von hier importiert, nie direkt ausgeführt.

## Warum ein Notebook und keine Skripte

Relative Pfade wie `data/interim/befunde/<Buch>` werden gegen das
**Arbeitsverzeichnis** aufgelöst, und wer das setzt, hängt am Werkzeug:

| Start über | Arbeitsverzeichnis |
|---|---|
| Jupyter-Kernel | Ordner der Notebook-Datei |
| VS Code, Python-Datei | Workspace-Ordner (oft eine Ebene höher) |

Das Notebook liegt im Projekt-Root und nimmt beim Start einmalig
`python/pdf_extraction/` in den Python-Suchpfad auf. Eine Installation der
eigenen Module ist dadurch nicht nötig. Die Module bleiben frei von
Bootstrap-Logik — fachliche Pfade kommen weiterhin aus `pfade.py`.

**Wer eine `.py`-Datei doch direkt startet, bekommt das alte Verhalten
zurück.** Abschnitt 0 macht das sichtbar, statt es stillschweigend
zurechtzurücken.

## Ablauf

| Abschnitt | Inhalt | Modell |
|---|---|---|
| 0 | Umgebung prüfen | – |
| 1 | Eingangsprüfung: der ganze Bestand vor jeder Verarbeitung | – |
| 2 | Dokumentenlauf: alle Dokumente in einem Aufruf | ONNX + VLM |
| 3 | Stufe 1+2 im Detail: ein einzelnes Buch zur Kontrolle | ONNX + VLM |
| 4 | Sichtung: was steht in den Befunden? | – |
| 5 | Nachlauf für abgeschnittene Blöcke | VLM |
| 6 | Stufe 4a im Detail: kanonisches Dokument und Markdown für ein Buch | – |
| 7 | Kontrolle | – |

---
## 0. Umgebung prüfen

Erst nachsehen, dann rechnen. Diese Zelle stellt nichts richtig, sie meldet
nur — ein falsches Arbeitsverzeichnis soll auffallen und nicht kaschiert
werden.

In [ ]:
from pathlib import Path
import sys

# Notebook liegt im Projekt-Root; eigene OCR-Module liegen unter
# python/pdf_extraction/ und werden installationsfrei eingebunden.
PROJECT_ROOT = Path.cwd().resolve()
MODULE_DIR = PROJECT_ROOT / "python" / "pdf_extraction"

if not MODULE_DIR.is_dir():
    raise RuntimeError(
        "OCR-Modulordner nicht gefunden: "
        f"{MODULE_DIR}\n"
        "Erwartete Struktur: <Projekt-Root>/python/pdf_extraction/"
    )

module_dir_str = str(MODULE_DIR)
if module_dir_str not in sys.path:
    sys.path.insert(0, module_dir_str)

import pfade

# --- Anzupassen
BUCH = None    # None: automatisch aus dem Scan unten (nur bei eindeutigem
               # Fund); bei mehreren Dokumenten hier von Hand einen Stamm setzen.
LMS  = "http://localhost:1234/v1"

print("Arbeitsverzeichnis:", Path.cwd())
print("Python            :", sys.version.split()[0], "\n")

# Scan über data/raw: legt BUCH automatisch fest, wenn die Auswahl eindeutig
# ist, statt den Namen blind vorzugeben oder von Hand raten zu lassen.
gefunden = (sorted(p.stem for p in pfade.RAW.iterdir()
                   if p.is_file() and p.suffix.lower() == ".pdf")
            if pfade.RAW.exists() else [])
print(f"Dokumente unter {pfade.RAW}: {gefunden or '(keine – oder der Ordner fehlt)'}")

if BUCH is None:
    if len(gefunden) == 1:
        BUCH = gefunden[0]
        print(f"  -> BUCH automatisch auf {BUCH!r} gesetzt (einzig gefundenes Dokument).")
    elif gefunden:
        BUCH = gefunden[0]
        print(f"  ! {len(gefunden)} Dokumente gefunden – BUCH oben von Hand auf eines "
              f"davon setzen. Vorläufig verwendet: {BUCH!r}.")
    else:
        BUCH = "Buch"
        print("  ! keine Dokumente unter data/raw gefunden – BUCH bleibt Platzhalter.")
elif BUCH not in gefunden:
    print(f"  ! BUCH = {BUCH!r} ist keines der gefundenen Dokumente.")
print()

PDF = pfade.quelle(BUCH)
ONNX = pfade.ONNX_STANDARD
BEFUNDE = pfade.BEFUNDE
AUSSCHNITTE = pfade.AUSSCHNITTE

for beschriftung, pfad in [("PDF", PDF), ("ONNX-Modell", ONNX),
                           ("Befunde", pfade.befund_ordner(BUCH)),
                           ("Ausschnitte", pfade.ausschnitt_ordner(BUCH))]:
    zustand = "vorhanden" if pfad.exists() else "fehlt"
    zusatz = ""
    if pfad.is_dir():
        zusatz = f"  ({len(list(pfad.glob('*'))) } Einträge)"
    print(f"  {beschriftung:14s} {zustand:10s} {pfad}{zusatz}")

# Liegt data/ versehentlich eine Ebene höher? Häufigster Fall, wenn eine
# .py-Datei direkt gestartet wurde (dann zeigt cwd auf den falschen Ordner).
fremd = Path("..") / pfade.WURZEL
if fremd.exists() and not pfade.WURZEL.exists():
    print(f"  ! {pfade.WURZEL!s} liegt eine Ebene höher: {fremd.resolve()}")

In [ ]:
# Module aus python/pdf_extraction laden. autoreload, damit Änderungen an den .py-Dateien sofort
# wirken, ohne den Kernel neu zu starten.
%load_ext autoreload
%autoreload 2

import pfade, schema, layout, erkennung, stapel, kanonisch, sichtung, lauf
from schema import Stufe, Strom, befund_laden, befund_pfad

print("Schema-Version:", schema.SCHEMA_VERSION)
print("Klassenabbildung:", schema.selbsttest_docling())

---
## 1. Eingangsprüfung — der ganze Bestand

SR-01 bis SR-05: der Eingangsbestand wird vollständig erfasst, Namen und
technische Öffnung geprüft, **bevor** irgendetwas verarbeitet wird. Alle
Beanstandungen erscheinen gemeinsam; ein beanstandeter Name wird genannt,
nicht selbst geändert (INV-4).

In [ ]:
dateien = lauf.erfassen()
print(f"{len(dateien)} Datei(en) unter {pfade.RAW}")

beanstandungen = lauf.pruefen(dateien)
if beanstandungen:
    print(f"\n{len(beanstandungen)} Datei(en) beanstandet — nichts wird verarbeitet:")
    for pfad in sorted(beanstandungen):
        for grund in beanstandungen[pfad]:
            print(f"  {pfad.name}: {grund}")
else:
    print("Keine Beanstandung. Der Bestand kann verarbeitet werden.")

---
## 2. Dokumentenlauf — alle Dokumente in einem Aufruf

SR-06, SR-07, SR-09: `verarbeite_alle` prüft zuerst (Abschnitt 1) und hält
bei jeder Beanstandung an. Danach läuft jedes Dokument nacheinander durch
den bestehenden Stufenlauf; ein gescheitertes Dokument wird vermerkt und
übersprungen, ein bereits abgeschlossenes gar nicht erst angerührt.

In [ ]:
# Ist in LM Studio das richtige Modell geladen? Ein winziges Bild klärt in
# einer Sekunde, was ein Buchlauf sonst auf jeder Seite wiederholt.
from erkennung import Erkenner

ERKENNER = Erkenner(url=LMS)
print(ERKENNER.pruefe() if hasattr(ERKENNER, "pruefe")
      else f"Modell: {ERKENNER.modell_id}")

In [ ]:
bericht = lauf.verarbeite_alle(erkenner=ERKENNER, zeige_fortschritt=False)
print(bericht.zusammenfassung())

---
## 3. Stufe 1 und 2 im Detail — ein einzelnes Buch

Abschnitt 2 deckt den ganzen Bestand ab; diese Zellen sind für die genaue
Kontrolle eines einzelnen Buchs (`BUCH`, oben eingestellt) gedacht — etwa
um eine Seite als Overlay zu sehen, bevor ein ganzes Buch läuft.

Der Lauf ist **stufenweise**: erst alle Seiten durch die Layout-Erkennung,
dann alle durch das VLM. Detektor und Sprachmodell sind damit nie
gleichzeitig geladen, die ONNX-Sitzung wird einmal geöffnet.

Fertige Seiten werden übersprungen (A2), gescheiterte übersprungen und
vermerkt (A3), Zeit und Modellversion je Seite festgehalten (A4).

**Erwartete Dauer:** Stufe 1 rund 0,4 s je Seite, Stufe 2 rund 11 s je
Seite. Für 222 Seiten also anderthalb Minuten plus vierzig.

In [ ]:
# Erst nur das Layout - kostet Sekunden und braucht kein LM Studio.
lauf_einzelbuch = stapel.verarbeite_buch(PDF, buch=BUCH, bis=Stufe.LAYOUT,
                                         onnx=ONNX, wurzel=BEFUNDE,
                                         ausschnitt_wurzel=AUSSCHNITTE)

In [ ]:
# Kontrolle vor dem teuren Teil: eine Seite als Overlay ansehen.
from IPython.display import Image, display
import cv2

SEITE = 0
befund = befund_laden(befund_pfad(BEFUNDE, BUCH, SEITE))
bild, _ = layout.seite_rendern(PDF, seite=SEITE)

ziel = pfade.kontrolle(BUCH, SEITE)
ziel.parent.mkdir(parents=True, exist_ok=True)

farben = {Strom.HAUPT: (9, 105, 218), Strom.MARGINALIE: (23, 138, 63),
          Strom.BOILERPLATE: (130, 130, 130), Strom.APPARAT: (191, 121, 15)}
leinwand = bild.copy()[:, :, ::-1].copy()
for blk in befund.bloecke:
    bx, f = blk.bbox, farben[blk.strom]
    cv2.rectangle(leinwand, (int(bx.x0), int(bx.y0)), (int(bx.x1), int(bx.y1)),
                  (f[2], f[1], f[0]), 2)
    cv2.putText(leinwand, f"{blk.lese_index}:{blk.pp_label}",
                (int(bx.x0) + 3, max(14, int(bx.y0) - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (f[2], f[1], f[0]), 1, cv2.LINE_AA)
cv2.imwrite(str(ziel), leinwand)
display(Image(str(ziel), width=760))

In [ ]:
# Jetzt die Erkennung. Bei einem ganzen Buch: Laufzeit einplanen.
lauf_einzelbuch = stapel.verarbeite_buch(PDF, buch=BUCH, bis=Stufe.ERKANNT,
                                         onnx=ONNX, wurzel=BEFUNDE,
                                         ausschnitt_wurzel=AUSSCHNITTE,
                                         erkenner=ERKENNER, zeige_fortschritt=False)
print(lauf_einzelbuch.zusammenfassung())

In [ ]:
# Was ist noch offen? Beantwortet A2, ohne etwas zu rechnen.
offen = stapel.offene_seiten(PDF, buch=BUCH, bis=Stufe.ERKANNT, wurzel=BEFUNDE)
print(len(offen), "offene Seiten", offen[:20])

---
## 4. Sichtung

Liest ausschließlich. Beantwortet mit Zahlen aus dem eigenen Korpus, was
sonst Vermutung bliebe: welche Klassen vorkommen, ob OTSL parsebar ist,
wie oft Detektionen ineinanderliegen, wie viele Seiten Überschriften
tragen.

In [ ]:
befunde = sichtung.befunde_lesen(BEFUNDE, BUCH)
sichtung.sichten(befunde)

---
## 5. Nachlauf für abgeschnittene Blöcke

`max_tokens` ist eine Sicherung gegen Wiederholungsschleifen, aber ein zu
knapper Deckel schneidet **still** ab: die Ausgabe liest sich flüssig und
hört einfach früher auf. Das Feld `finish_reason` der Antwort verrät es,
und die Erkennung schreibt es seit der Korrektur als Warnung in den Befund.

Diese Zelle sucht Blöcke, die verdächtig nahe am Deckel liegen, und lässt
nur deren Seiten erneut laufen — Minuten statt einer weiteren Stunde.

In [ ]:
GRENZE = 2800     # Zeichen; 1024 Tokens sind für Deutsch grob 3200

lang = [(len(b.text), bf.seite, b.id, b.pp_label, b.text[-50:].replace("\n", " "))
        for bf in befunde for b in bf.bloecke
        if b.text and len(b.text) > GRENZE]

for laenge, seite, bid, label, ende in sorted(lang, reverse=True)[:15]:
    print(f"{laenge:5d} S{seite:3d} #{bid:2d} {label:16s} …{ende!r}")

# Enden sie mitten im Wort, ist es der Deckel und kein Textmerkmal.
verdaechtig = sorted({s for _, s, _, _, _ in lang})
print(f"\n{len(lang)} Blöcke über {GRENZE} Zeichen auf {len(verdaechtig)} Seiten")
print(verdaechtig)

In [ ]:
# Nachlauf. neu=True ist nötig, sonst überspringt A2 genau diese Seiten.
if verdaechtig:
    bericht_nachlauf = stapel.stufe2_lauf(PDF, BUCH, verdaechtig, ERKENNER,
                                          wurzel=BEFUNDE, ausschnitt_wurzel=AUSSCHNITTE,
                                          neu=True, zeige_fortschritt=False)
    print(bericht_nachlauf.zeile())

# Ist danach noch etwas abgeschnitten?
uebrig = [(bf.seite, w) for bf in sichtung.befunde_lesen(BEFUNDE, BUCH)
          for w in bf.warnungen if "Token-Deckel" in w]
print(f"\n{len(uebrig)} Blöcke weiterhin am Deckel")
for seite, w in uebrig[:10]:
    print(f"  S{seite:3d}  {w}")

---
## 6. Stufe 4a im Detail — das kanonische Dokument

Übergang vom Arbeitsformat (Beobachtung, mit Widersprüchen) zum
`DoclingDocument` (Entscheidung: eine Region, ein Label, ein Text).
Vollständig deterministisch, ohne Sprachmodell. Abschnitt 2 hat das für
den ganzen Bestand schon erledigt; hier zur genauen Kontrolle für `BUCH`.

Seitengrenzen werden nirgends aufgehoben: was auf einer Seite steht, bleibt
dort, damit jede Aussage rückwärts einer Seite zurechenbar bleibt.

In [ ]:
dok, bericht = kanonisch.buch_umwandeln(BUCH, wurzel=BEFUNDE)

---
## 7. Kontrolle

Die billigste Prüfung ist ein Blick auf das Ergebnis. Wenn sich der
Fließtext an einer Kapitelgrenze flüssig liest und die Gliederungsebenen
stimmen, sind Lesereihenfolge, Erkennung und Übergang zugleich in Ordnung.

In [ ]:
# Gliederung: tragen die Ebenen?
from docling_core.types.doc import DocItemLabel

for item, _ in dok.iterate_items():
    if item.label is DocItemLabel.SECTION_HEADER:
        print(f"{'  ' * (item.level - 1)}{item.level}  {item.text[:70]}")

In [ ]:
# Ein Ausschnitt des Markdown, ab einer beliebigen Stelle.
text = pfade.dokument_md(BUCH).read_text(encoding="utf-8")
print(f"{len(text)} Zeichen, {text.count(chr(10)) + 1} Zeilen\n")
print(text[:3000])

In [ ]:
# Offene Bildunterschriften: die Arbeitsliste für den VLM-Pass in 4b.
print(len(bericht.offene_unterschriften), "offene Zuordnungen")
for seite, uid, oids in bericht.offene_unterschriften[:15]:
    bf = befund_laden(befund_pfad(BEFUNDE, BUCH, seite))
    u = next(b for b in bf.bloecke if b.id == uid)
    print(f"  S{seite:3d}  #{uid:2d} {(u.text or '')[:52]!r}  -> Kandidaten {oids}")

In [ ]:
# Alle Warnungen des Übergangs.
print("\n".join(bericht.warnungen) or "keine")

---
## Nicht in dieser Etappe

| # | Punkt | Warum |
|---|---|---|
| A7 | Absatz über Seitengrenze zusammenfassen | Seitengrenzen bleiben bestehen, damit die Seitenzuweisung erhalten bleibt |
| A13 | Inline-Formel zurück in den Satz | die Position im erkannten Text ist nicht bekannt |
| 4b | Bildunterschriften zuordnen, Kästen typisieren, Zahlentabellen zusammenfassen | keine eindeutig richtige Form; braucht ein Bildmodell |

Die Trennung ist der Kern: Deterministisches und Generatives
auseinanderzuhalten hält den prüfbaren Teil prüfbar.